# Normalización del dataset de clima de Galicia

### 1. Importar librerias necesarias

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import os

from pathlib import Path
from difflib import get_close_matches

### 2 - Carga del dataset a normalizar

In [2]:
# Definir la ruta del dataset
dataset_path = r'C:\00 - Proyecto Incendios Galicia - END\data\01 - raw\03 - meteorologia\01 - clima galicia.csv'

# Cargar el archivo CSV directamente
df = pd.read_csv(dataset_path)

print(f'Filas cargadas: {len(df)}')
print('Columnas disponibles:', list(df.columns))
display(df.head())

# Mostrar el número de municipios únicos en la columna 'municipio'
print(f"Municipios únicos en el dataset: {df['municipio'].nunique()}")

C:\Users\Jacinto\AppData\Local\Temp\ipykernel_20712\3097029388.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(dataset_path)


Filas cargadas: 2614336
Columnas disponibles: ['time', 'tavg', 'tmin', 'tmax', 'prcp', 'municipio', 'proveniente_de']


,time,tavg,tmin,tmax,prcp,municipio,proveniente_de
0,2000-01-01,NaN,-5.8,9.6,0.0,ABADÍN,NaN
1,2000-01-02,NaN,-4.4,6.2,0.0,ABADÍN,NaN
2,2000-01-03,NaN,-4.4,11.2,0.0,ABADÍN,NaN
3,2000-01-04,NaN,5.8,9.6,0.3,ABADÍN,NaN
4,2000-01-05,NaN,3.2,11.2,0.0,ABADÍN,NaN


Municipios únicos en el dataset: 312


### 3 - Visualización y exploración inicial

In [3]:
# Ver las primeras filas del dataset
df.head()

# Ver información general del dataset
df.info()

# Ver estadísticas básicas de las columnas numéricas
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2614336 entries, 0 to 2614335
Data columns (total 7 columns):
 #   Column          Dtype  
---  ------          -----  
 0   time            object 
 1   tavg            float64
 2   tmin            float64
 3   tmax            float64
 4   prcp            float64
 5   municipio       object 
 6   proveniente_de  object 
dtypes: float64(4), object(3)
memory usage: 139.6+ MB


,tavg,tmin,tmax,prcp
count,2.011178e+06,2.599615e+06,2.599607e+06,2.581788e+06
mean,1.415848e+01,9.088965e+00,1.951998e+01,3.179351e+00
std,5.323878e+00,5.115649e+00,6.794467e+00,7.577437e+00
min,-4.300000e+00,-1.020000e+01,-1.500000e+00,0.000000e+00
25%,1.030000e+01,5.600000e+00,1.420000e+01,0.000000e+00
50%,1.400000e+01,9.500000e+00,1.880000e+01,0.000000e+00
75%,1.810000e+01,1.300000e+01,2.410000e+01,2.400000e+00
max,3.270000e+01,2.550000e+01,4.410000e+01,1.719000e+02


### 4 - Cargar el dataset limpio de municipios de Galicia

Vamos a cargar la tabla de municipios de Galicia que creamos antes. Usaremos estos nombres como referencia para normalizar los municipios del dataset de incendios.

In [4]:
# Ruta del archivo de municipios limpios
municipios_path = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\01 - municipios\01 - municipios normalizados.csv'

# Cargar el archivo Excel de municipios
df_municipios = pd.read_csv(municipios_path)

# Mostrar las primeras filas para comprobar que se ha cargado bien
display(df_municipios.head())

# Mostrar el número de municipios únicos en el dataset de municipios
print(f"Municipios únicos en el dataset de municipios: {df_municipios['municipio'].nunique()}")

,municipio,comarca,provincia,altitud,superficie,poblacion,densidad
0,a arnoia,comarca del ribeiro,ourense,76,"20,69",1.000,"48,33"
1,a baña,barcala,a coruña,297,"98,19",3.450,"35,14"
2,a bola,comarca de tierra de celanova,ourense,510,"34,9",1.156,"33,12"
3,a capela,comarca del eume,a coruña,NaN,58,1.232,"21,24"
4,a cañiza,comarca de paradanta,pontevedra,570,"105,04",5.180,"49,31"


Municipios únicos en el dataset de municipios: 315


### 5 - Normalización automática de municipios (optimizada por mapeo único y orden invertido)

In [5]:
ruta_txt = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\03 - meteorologia'
os.makedirs(ruta_txt, exist_ok=True)
# Definir las columnas a usar
col_municipio = 'municipio'  # columna a normalizar en el dataset principal
col_ref = 'municipio'        # columna de referencia en el dataset de municipios
df_ref = df_municipios       # referencia oficial

# Función para normalizar nombres eliminando tildes, mayúsculas, signos y espacios extra
def normalizar_nombre(nombre):
    if pd.isnull(nombre):
        return ''
    nombre = str(nombre).strip().lower()
    nombre = ''.join(c for c in unicodedata.normalize('NFD', nombre) if unicodedata.category(c) != 'Mn')
    nombre = nombre.replace('-', ' ').replace(',', '').replace('.', '')
    nombre = ' '.join(nombre.split())
    return nombre

# Diccionario de referencia normalizada para búsqueda rápida
ref_norm = {normalizar_nombre(x): x for x in df_ref[col_ref].dropna().unique()}
ref_norm_keys = set(ref_norm.keys())

# 1. Obtener todos los valores únicos del dataset a normalizar y de la referencia
df[col_municipio] = df[col_municipio].astype(str)
municipios_unicos = set(x for x in df[col_municipio].unique() if isinstance(x, str) and x.strip())
municipios_referencia = set(df_ref[col_ref].dropna().unique())

# 2. Crear mapeo: municipio original -> municipio normalizado (o sugerido, o pendiente)
mapeo = {}
pendientes = []
for m in municipios_unicos:
    clave = normalizar_nombre(m)
    if clave in ref_norm:
        mapeo[m] = ref_norm[clave]
        continue
    sugerencias = get_close_matches(clave, ref_norm_keys, n=1, cutoff=0.8)
    if sugerencias:
        mapeo[m] = ref_norm[sugerencias[0]]
        continue
    # Probar a invertir el orden de las palabras si hay exactamente dos
    partes = clave.split()
    if len(partes) == 2:
        invertido = ' '.join(partes[::-1])
        if invertido in ref_norm:
            mapeo[m] = ref_norm[invertido]
            continue
        sugerencias_inv = get_close_matches(invertido, ref_norm_keys, n=1, cutoff=0.8)
        if sugerencias_inv:
            mapeo[m] = ref_norm[sugerencias_inv[0]]
            continue
    # Si no se encuentra nada, dejar el original y marcar como pendiente
    mapeo[m] = m
    pendientes.append(m)

# 3. Aplicar el mapeo a todo el dataset
df['Municipio_normalizado'] = df[col_municipio].map(mapeo)

# 4. Diagnóstico de diferencias entre dataset y referencia
municipios_normalizados = set(df['Municipio_normalizado'].dropna().unique())
faltan_en_dataset = municipios_referencia - municipios_normalizados
sobran_en_dataset = municipios_normalizados - municipios_referencia

print(f"Municipios únicos en el dataset de referencia: {len(municipios_referencia)}")
print(f"Municipios únicos normalizados en el dataset principal: {len(municipios_normalizados)}")
if faltan_en_dataset:
    print(f"Municipios de la referencia que NO aparecen en el dataset principal: {faltan_en_dataset}")
else:
    print("Todos los municipios de la referencia están presentes en el dataset principal.")
if sobran_en_dataset:
    print(f"Municipios en el dataset principal que NO están en la referencia: {sobran_en_dataset}")
else:
    print("No hay municipios extra en el dataset principal.")

# 5. Exportar el diccionario de correspondencias y los pendientes
import csv
archivo_diccionario = os.path.join(ruta_txt, 'diccionario_normalizacion_final.txt')
with open(archivo_diccionario, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['original', 'normalizado'])
    for k, v in sorted(mapeo.items()):
        writer.writerow([k, v])

archivo_pendientes = os.path.join(ruta_txt, 'municipios_no_normalizados_final.txt')
with open(archivo_pendientes, 'w', encoding='utf-8') as f:
    for m in pendientes:
        f.write(f'{m}\n')

print(f"Municipios únicos originales en el dataset principal: {len(municipios_unicos)}")
print(f"Municipios normalizados automáticamente: {len(mapeo) - len(pendientes)}")
print(f"Municipios pendientes de normalizar: {len(pendientes)}")
print(f'Diccionario de normalización exportado a {archivo_diccionario}')
print(f'Listado de pendientes exportado a {archivo_pendientes}')

Municipios únicos en el dataset de referencia: 315
Municipios únicos normalizados en el dataset principal: 311
Municipios de la referencia que NO aparecen en el dataset principal: {'cerceda', 'sarria', 'cervantes', 'porquera', 'carballo', 'sobrado', 'a rúa', 'muros', 'cee'}
Municipios en el dataset principal que NO están en la referencia: {'SARRIA_(LUGO)', 'CERVANTES_(LUGO)', 'CEE_(A_CORUÑA)', 'SOBRADO_(A_CORUÑA)', 'MUROS_(A_CORUÑA)'}
Municipios únicos originales en el dataset principal: 312
Municipios normalizados automáticamente: 307
Municipios pendientes de normalizar: 5
Diccionario de normalización exportado a C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\03 - meteorologia\diccionario_normalizacion_final.txt
Listado de pendientes exportado a C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\03 - meteorologia\municipios_no_normalizados_final.txt


### 5.1 Desdoblar "cerdedo-cotobade" en "cerdedo" y "cotobade"

In [6]:
# Esta celda busca todas las filas cuyo municipio normalizado es 'cercedo-cotobade' y las duplica,
# creando dos nuevas filas: una para 'cercedo' y otra para 'cotobade', manteniendo el resto de datos.

# Buscar filas normalizadas como 'cercedo-cotobade'
mask = df['Municipio_normalizado'].str.lower().str.strip() == 'cerdedo-cotobade'
filas_fusionadas = df[mask]

# Si existen, duplicar para 'cercedo' y 'cotobade'
if not filas_fusionadas.empty:
    nuevas_filas = []
    for _, row in filas_fusionadas.iterrows():
        for nuevo_muni in ['cerdedo', 'cotobade']:
            nueva = row.copy()
            nueva['Municipio_normalizado'] = nuevo_muni
            nuevas_filas.append(nueva)
    df = pd.concat([df, pd.DataFrame(nuevas_filas)], ignore_index=True)
    print(f"Se han añadido {2*len(filas_fusionadas)} filas para 'cerdedo' y 'cotobade'.")
else:
    print("No se encontraron filas con municipio 'cerdedo-cotobade'.")

# Registrar nuevamente los municipios únicos y comparar con la referencia
municipios_normalizados = set(df['Municipio_normalizado'].dropna().unique())
faltan_en_dataset = municipios_referencia - municipios_normalizados
sobran_en_dataset = municipios_normalizados - municipios_referencia

print(f"\nTras desdoblar 'cerdedo-cotobade':")
print(f"Municipios únicos normalizados en el dataset principal: {len(municipios_normalizados)}")
if faltan_en_dataset:
    print(f"Municipios de la referencia que NO aparecen en el dataset principal: {faltan_en_dataset}")
else:
    print("Todos los municipios de la referencia están presentes en el dataset principal.")
if sobran_en_dataset:
    print(f"Municipios en el dataset principal que NO están en la referencia: {sobran_en_dataset}")
else:
    print("No hay municipios extra en el dataset principal.")

Se han añadido 16802 filas para 'cerdedo' y 'cotobade'.

Tras desdoblar 'cerdedo-cotobade':
Municipios únicos normalizados en el dataset principal: 311
Municipios de la referencia que NO aparecen en el dataset principal: {'cerceda', 'sarria', 'cervantes', 'porquera', 'carballo', 'sobrado', 'a rúa', 'muros', 'cee'}
Municipios en el dataset principal que NO están en la referencia: {'SARRIA_(LUGO)', 'CERVANTES_(LUGO)', 'CEE_(A_CORUÑA)', 'SOBRADO_(A_CORUÑA)', 'MUROS_(A_CORUÑA)'}


### 5.2 Sustituir los valores de la columna municipio por los correctos

In [7]:
# Sustituir la columna 'municipio' por los valores corregidos y eliminar 'Municipio_normalizado'
df['municipio'] = df['Municipio_normalizado']
df = df.drop(columns=['Municipio_normalizado'])

### 6. Exportar el dataset final con municipios normalizados

A continuación se exporta el dataframe resultante, que contiene todos los municipios normalizados y desdoblados, a archivos Excel y CSV para su uso posterior.

In [ ]:

import os

ruta_export = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\03 - meteorologia'
os.makedirs(ruta_export, exist_ok=True)
archivo_export = os.path.join(ruta_export, '01 - clima municipios normalizados.csv')

df.to_csv(archivo_export, index=False, encoding='utf-8')
print(f'Dataset final exportado como {archivo_export}')

Dataset final exportado como C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\03 - meteorologia\clima galicia municipios normalizados.csv


In [ ]:
# Comprobación final: presencia y recuento de los tres municipios clave

# Definir la ruta del dataset
dataset_path = r'C:\00 - Proyecto Incendios Galicia - END\data\02 - Municipio normalizado\03 - meteorologia\01 - clima municipios normalizados.csv'



df_check = pd.read_csv(dataset_path)

for muni in ['cerdedo', 'cotobade', 'cercedo-cotobade']:
    count = (df_check['municipio'].str.lower().str.strip() == muni).sum()
    print(f"{muni}: {count} registros")

C:\Users\Jacinto\AppData\Local\Temp\ipykernel_20712\3826287170.py:8: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_check = pd.read_csv(dataset_path)


cerdedo: 16802 registros
cotobade: 16802 registros
cercedo-cotobade: 0 registros
